## LLM - Assistente de recomendações de jogos de PC
Modelo com base nas descrições e análises de jogos da Steam, com chat que sugere jogos ao usuário.   
O pipeline utiliza LangChain e Transformers, e um dataset Kaggle consultado por HuggingFace.

In [1]:
# Instalações
%pip install langchain langchain-community transformers accelerate sentence-transformers faiss-cpu datasets deep-translator kagglehub[hf-datasets] -U langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Success

In [13]:
from transformers import logging

# Only report critical errors, hiding warnings and info logs
logging.set_verbosity_error()

In [2]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Datasets
## Nesse projeto usei o Steam Store Games, dataset com 27000 jogos da Steam
## https://www.kaggle.com/datasets/nikdavis/steam-store-games
## Carga usando Hugging Face, ao invés de Pandas, pra poupar memória

dataset_principal = kagglehub.load_dataset(
  KaggleDatasetAdapter.HUGGING_FACE,
  "nikdavis/steam-store-games",
  "steam.csv"
)

print("Dataset Principal:", dataset_principal)

dataset_descricoes = kagglehub.load_dataset(
    KaggleDatasetAdapter.HUGGING_FACE,
    "nikdavis/steam-store-games",
    "steam_description_data.csv"
)

print("Dataset Descrições:", dataset_descricoes)

/tmp/ipykernel_4167/1782193740.py:9: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  dataset_principal = kagglehub.load_dataset(


Using Colab cache for faster access to the 'steam-store-games' dataset.
Dataset Principal: Dataset({
    features: ['appid', 'name', 'release_date', 'english', 'developer', 'publisher', 'platforms', 'required_age', 'categories', 'genres', 'steamspy_tags', 'achievements', 'positive_ratings', 'negative_ratings', 'average_playtime', 'median_playtime', 'owners', 'price'],
    num_rows: 27075
})


/tmp/ipykernel_4167/1782193740.py:17: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  dataset_descricoes = kagglehub.load_dataset(


Using Colab cache for faster access to the 'steam-store-games' dataset.
Dataset Descrições: Dataset({
    features: ['steam_appid', 'detailed_description', 'about_the_game', 'short_description'],
    num_rows: 27334
})


In [3]:
# Fazendo o join entre os dois datasets, pra incorporar as descrições ao dataset principal

dict_descricoes = {
    linha["steam_appid"]: {
        "short": linha["short_description"]
    }
    for linha in dataset_descricoes
}

dict_descricoes

{10: {'short': "Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role."},
 20: {'short': 'One of the most popular online action games of all time, Team Fortress Classic features over nine character classes -- from Medic to Spy to Demolition Man -- enlisted in a unique style of online team warfare. Each character class possesses unique weapons, items, and abilities, as teams compete online in a variety of game play modes.'},
 30: {'short': 'Enlist in an intense brand of Axis vs. Allied teamplay set in the WWII European Theatre of Operations. Players assume the role of light/assault/heavy infantry, sniper or machine-gunner class, each with a unique arsenal of historical weaponry at their disposal. Missions are based on key historic

In [4]:
def adicionar_descricoes(linha):
    id_jogo = linha["appid"]

    # Busca o pacote de descricoes no dicionario. Se nao achar, retorna dicionario vazio
    dados_desc = dict_descricoes.get(id_jogo, {})

    # Cria a nova coluna de descrição na linha atual
    linha["short_description"] = dados_desc.get("short", "")

    return linha

# Aplica a funcao paralelizada no dataset principal
dataset_completo = dataset_principal.map(adicionar_descricoes)

# Exibe a primeira linha para validar
print("Dataset enriquecido com a coluna de descrição")
print(dataset_completo[0])

Map:   0%|          | 0/27075 [00:00<?, ? examples/s]

Dataset enriquecido com a coluna de descrição
{'appid': 10, 'name': 'Counter-Strike', 'release_date': '2000-11-01', 'english': 1, 'developer': 'Valve', 'publisher': 'Valve', 'platforms': 'windows;mac;linux', 'required_age': 0, 'categories': 'Multi-player;Online Multi-Player;Local Multi-Player;Valve Anti-Cheat enabled', 'genres': 'Action', 'steamspy_tags': 'Action;FPS;Multiplayer', 'achievements': 0, 'positive_ratings': 124534, 'negative_ratings': 3339, 'average_playtime': 17612, 'median_playtime': 317, 'owners': '10000000-20000000', 'price': 7.19, 'short_description': "Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role."}


In [75]:
from langchain_core.documents import Document

documentos = []

# Amostragem inicial pra agilizar, removida depois dos testes
# subset = dataset_completo.select(range(100))
subset = dataset_completo

for linha in subset:
    # Concatenando as informações do dataset
    texto_consolidado = (
        f"Game: {linha['name']} | Developer: {linha['developer']} | Genres: {linha['genres']}\n"
        f"Details: {linha['short_description']}"
        f"Positive ratings: {linha['positive_ratings']}"
    )

    # Metadados pra aplicar filtros
    metadados = {
        "appid": linha["appid"],
        "name": linha["name"],
        "genres": linha["genres"],
        "positive_ratings": linha["positive_ratings"]
    }

    doc = Document(page_content=texto_consolidado, metadata=metadados)
    documentos.append(doc)

print(f"{len(documentos)} documentos criados com sucesso.")

27075 documentos criados com sucesso.


In [76]:
# Usando o Text Splitter pra separar os textos longos em chunks legíveis
import langchain_text_splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,       # Quantidade máxima de caracteres por bloco
    chunk_overlap=50      # Caracteres repetidos entre blocos vizinhos para manter o contexto
)

documentos_divididos = text_splitter.split_documents(documentos)
print(f"Total de chunks gerados: {len(documentos_divididos)}")

Total de chunks gerados: 32078


In [77]:
# Modelo de embedding do Hugging Face
## Usei o all-MiniLM pois é leve e gratuito

from langchain_community.embeddings import HuggingFaceEmbeddings

# Carregando o modelo de representação vetorial
modelo_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [78]:
# Banco vetorial FAISS

from langchain_community.vectorstores import FAISS

# Criando e indexando a base vetorial em memória
banco_vetorial = FAISS.from_documents(documentos_divididos, modelo_embeddings)
print("Banco vetorial FAISS criado e indexado com sucesso.")

Banco vetorial FAISS criado e indexado com sucesso.


In [79]:
# Funções de tradução - inglês-português e português-inglês
from deep_translator import GoogleTranslator

def traduzir_pt_para_en(texto):
    # Traduz a pergunta do usuario para o idioma do dataset e do modelo
    tradutor = GoogleTranslator(source='pt', target='en')
    return tradutor.translate(texto)

def traduzir_en_para_pt(texto):
    # Traduz a resposta final do modelo de volta para o usuario
    tradutor = GoogleTranslator(source='en', target='pt')
    return tradutor.translate(texto)

In [80]:
# Configurando o LLM, criando o template de prompt e criando a chain
from transformers import pipeline
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Configuração da LLM
gerador = pipeline(
    task="text-generation",
    model="Qwen/Qwen2-1.5B-Instruct",
    max_new_tokens=250,
    return_full_text=False, # Impede que o modelo devolva o prompt de instrução junto com a resposta
)

llm = HuggingFacePipeline(pipeline=gerador)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [81]:
# Prompt template
template_recomendacao = """
You are a passionate and expert PC game recommender. Your goal is to suggest the best games based ONLY on the provided context.

Context (Available Games):
{contexto}

User Request:
{pergunta}

Instructions:
1. Recommend games from the context that best match the user's request, recommending popular games first.
2. If the exact genre isn't there, suggest the closest matches available in the context and explain why they fit.
3. Keep the tone friendly and exciting.
4. DO NOT invent games. If the context is completely unrelated, apologize and say you don't have good matches right now.

Recommendation:
"""

prompt = PromptTemplate.from_template(template_recomendacao)

# Chain
chain = prompt | llm | StrOutputParser()

In [89]:
# Pipeline de recomendação
def recomendar_jogos(pergunta_pt, banco_vetorial):
    print("Processando sua requisição...")

    # 1. Traduz a pergunta para inglês
    pergunta_en = traduzir_pt_para_en(pergunta_pt)

    # 2. Busca no banco vetorial
    documentos_brutos = banco_vetorial.similarity_search(pergunta_en, k=50)

    # 3. Ordenação corrigida
    documentos_ordenados = sorted(
        documentos_brutos,
        key=lambda doc: int(doc.metadata.get('positive_ratings', 0)),
        reverse=True
    )

    # Corta para os top 3 absolutos em popularidade dentro daquela rede de 50
    top_3_jogos = documentos_ordenados[:3]

    # 4. Extrai e junta o texto dos top 3 jogos recuperados
    contexto_en = "\n\n".join([doc.page_content for doc in top_3_jogos])

    # 5. Background para debugging
    print("\n--- INÍCIO DO CONTEXTO DO FAISS ---")
    for d in top_3_jogos:
        print(f"Jogo: {d.metadata.get('name', 'Desconhecido')} | Avaliações Positivas: {d.metadata.get('positive_ratings', 0)}")
    print("--- FIM DO CONTEXTO ---\n")

    # 6. Passa a pergunta e o contexto para a LLM via LangChain
    resposta_en = chain.invoke({"contexto": contexto_en, "pergunta": pergunta_en})

    # 7. Traduz a resposta final de volta para português
    if "Game recommended:" in resposta_en:
        resposta_en = resposta_en.split("Game recommended:")[-1]

    resposta_limpa_en = resposta_en.strip()
    resposta_pt = traduzir_en_para_pt(resposta_limpa_en)

    return resposta_pt

In [91]:
## Exemplos de recomendação
recomendar_jogos("Me recomende jogos do gênero survival", banco_vetorial)

Processando sua requisição...

--- INÍCIO DO CONTEXTO DO FAISS ---
Jogo: ARK: Survival Evolved | Avaliações Positivas: 145035
Jogo: Dying Light | Avaliações Positivas: 78192
Jogo: SURVIVAL | Avaliações Positivas: 3931
--- FIM DO CONTEXTO ---



'Com base na sua solicitação, recomendamos "ARK: Survival Evolved", pois se adapta perfeitamente ao seu interesse em jogos de sobrevivência devido aos seus enormes elementos multijogador, jogabilidade estilo RPG e gênero de ação e aventura. Possui avaliações positivas de mais de 70.000 jogadores.\n\nAlém disso, gostaria de recomendar "Dying Light", uma excelente escolha se você gosta de jogos de ação e sobrevivência em primeira pessoa, onde você pode procurar recursos, fabricar armas e enfrentar hordas de infectados em um mundo aberto pós-apocalíptico. Com uma avaliação positiva de 78.192, parece ser bastante bem recebido entre os jogadores.\n\nPor último, se você estiver interessado em uma experiência mais casual, “SURVIVAL” pode ser uma opção melhor. Este jogo oferece uma vertente multijogador e é descrito como um jogo de sobrevivência num mundo pós-apocalíptico, que pode agradar a quem gosta de jogar online com amigos ou estranhos. Ele também tem uma classificação positiva de 3.931.

In [90]:
## Exemplos de recomendação
recomendar_jogos("Me recomende jogos de ritmo", banco_vetorial)

Processando sua requisição...

--- INÍCIO DO CONTEXTO DO FAISS ---
Jogo: Geometry Dash | Avaliações Positivas: 52737
Jogo: Jazzpunk: Director's Cut | Avaliações Positivas: 4136
Jogo: BIT.TRIP Presents... Runner2: Future Legend of Rhythm Alien | Avaliações Positivas: 2615
--- FIM DO CONTEXTO ---



'Com base no contexto, recomendo “Geometry Dash” por ser um jogo de ação e plataforma com forte foco na jogabilidade rítmica, que atende à solicitação do usuário por jogos rítmicos. Possui avaliações positivas de mais de 50.000, indicando sua popularidade entre os jogadores que gostam desse tipo de jogo. Além disso, combina com a reputação do desenvolvedor como RobTop Games, conhecido por seu trabalho em outros jogos de ritmo, como “Space Invaders” e “The Legend of Zelda: Ocarina of Time”. Portanto, se você está procurando um jogo com recursos semelhantes e um histórico sólido, “Geometry Dash” pode ser uma ótima escolha. Divirta-se jogando!\n\nJogo: Bit.Trip Rhythm: Um jogo de plataforma | Desenvolvedor: Bit.Trip Inc. Gêneros: Quebra-cabeça;Indie\n\nJogo: Space Invaders: Modo Batalha | Desenvolvedor: Namco | Gêneros: Ação; Indie\n\nJogo: The Legend of Zelda: Ocarina of Time | Desenvolvedor: Nintendo | Gêneros: Ação; Indie\n\nJogo: Efeito Tetris: Edição Musical | Desenvolvedor: Supercel